# 09 — Saving & Loading Models

This notebook demonstrates the model-persistence workflow for the workshop GPT model.

The practical flow is:

```text
Rebuild architecture
      ↓
Load `.pth` checkpoint
      ↓
Restore learned weights
      ↓
Run inference
      ↓
Convert/save in Hugging Face format with SafeTensors
      ↓
Reload through `AutoModelForCausalLM`
```

The notebook therefore connects training checkpoints with reusable model artifacts.

## 0 — Setup

In [9]:
#@title 0) setup
%pip install -q torch torchtune tiktoken huggingface_hub

import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from huggingface_hub import hf_hub_download

import tiktoken
from torchtune.modules import (
    RMSNorm,
    RotaryPositionalEmbeddings,
    MultiHeadAttention,
    TransformerSelfAttentionLayer,
    TransformerDecoder,
    TiedLinear,
)

## 1 — Rebuild the Workshop Model Architecture

The architecture must match the checkpoint configuration before the saved weights
can be restored.

The configuration used here is:

```text
layers:        12
attention heads: 12
embedding size: 768
FFN hidden size: 3072
vocabulary size: 50304
context length: 1024
```

The model uses ReLU² in the MLP, RoPE positional embeddings, RMSNorm, and tied
input/output embeddings.

In [10]:
@dataclass
class GPTConfig:
    n_layer:    int   = 12
    n_head:     int   = 12
    n_embd:     int   = 768
    vocab_size: int   = 50304
    block_size: int   = 1024
    dropout:    float = 0.0      # no dropout at inference
    n_inner:    int   = 3072
    rope_theta: float = 10000.0

In [11]:
class ReluSquaredMLP(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.fc_in  = nn.Linear(dim, hidden_dim, bias=False)
        self.fc_out = nn.Linear(hidden_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc_out(F.relu(self.fc_in(x)).square()))


def build_model(c: GPTConfig) -> TransformerDecoder:
    head_dim = c.n_embd // c.n_head

    rope = RotaryPositionalEmbeddings(
        dim=head_dim, max_seq_len=c.block_size, base=c.rope_theta
    )
    attn = MultiHeadAttention(
        embed_dim=c.n_embd, num_heads=c.n_head, num_kv_heads=c.n_head,
        head_dim=head_dim,
        q_proj=nn.Linear(c.n_embd, c.n_embd, bias=False),
        k_proj=nn.Linear(c.n_embd, c.n_embd, bias=False),
        v_proj=nn.Linear(c.n_embd, c.n_embd, bias=False),
        output_proj=nn.Linear(c.n_embd, c.n_embd, bias=False),
        pos_embeddings=rope, max_seq_len=c.block_size, attn_dropout=c.dropout,
    )
    mlp = ReluSquaredMLP(c.n_embd, c.n_inner, c.dropout)
    layer = TransformerSelfAttentionLayer(
        attn=attn, mlp=mlp,
        sa_norm=RMSNorm(c.n_embd), mlp_norm=RMSNorm(c.n_embd),
    )
    tok_emb = nn.Embedding(c.vocab_size, c.n_embd)

    return TransformerDecoder(
        tok_embeddings=tok_emb, layers=layer, num_layers=c.n_layer,
        max_seq_len=c.block_size, num_heads=c.n_head, head_dim=head_dim,
        norm=RMSNorm(c.n_embd), output=TiedLinear(tok_emb),
    )

## 2 — Load a `.pth` Checkpoint from Hugging Face

The checkpoint contains the learned parameter state. The notebook rebuilds the
architecture first, downloads the checkpoint, extracts the state dictionary, and
loads it into the model.

The recorded run restored a model with **123,587,328 parameters**.

In [12]:
REPO_ID  = "JustinAngel/workshop-v1-pretraining"
FILENAME =  "workshop-v1-pretraining.pth"   # or "workshop-v1-instruct-tuned.pth"
DEVICE   = "cuda"

cfg   = GPTConfig()
model = build_model(cfg)

path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
ckpt = torch.load(path, map_location=DEVICE, weights_only=False)

state_dict = ckpt["model"] if "model" in ckpt else ckpt
model.load_state_dict(state_dict)
model = model.to(DEVICE).eval()

print(f"Loaded — {sum(p.numel() for p in model.parameters()):,} parameters")

Loaded — 123,587,328 parameters


## 3 — Verify the Restored Model with Text Generation

After loading, inference is used as a functional check that the restored model can
generate text.

Generation uses:

- GPT-2 tokenization
- temperature sampling
- Top-K filtering
- autoregressive next-token generation

In [13]:
enc = tiktoken.get_encoding("gpt2")

@torch.no_grad()
def generate(prompt: str, max_tokens=100, temperature=0.8, top_k=50):
    tokens = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=DEVICE)

    for _ in range(max_tokens):
        idx = tokens[:, -cfg.block_size:]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(idx)
        logits = logits[:, -1, :] / temperature
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")
        tok = torch.multinomial(F.softmax(logits, dim=-1), 1)
        tokens = torch.cat([tokens, tok], dim=1)
        if tok.item() == enc.eot_token:
            break

    return enc.decode(tokens[0].tolist())

In [14]:
for prompt in ["The meaning of life is", "Once upon a time there was"]:
    print(f">> {prompt}")
    print(generate(prompt))
    print()

>> The meaning of life is
The meaning of life is the same as life itself. There is a complete and comprehensive understanding of the physical and emotional state of the human being that can be found in all of the other major life forms.
The physical and emotional state of the human being is a unique and universal feature of all life. The physical and emotional state of the human being is a universal feature of all of the other major life forms.
The physical and emotional state of the human being is a universal feature of all life. If there is no

>> Once upon a time there was
Once upon a time there was a large crowd of people, and it was a very good time to visit. And they would come to the same place every day and ask for a drink of water and honey.
The Indians were angry and angry and threatened to kill the natives of the region if they did not leave. But the natives were scared because they did not want to leave their homes and they wanted to go back to their homeland. They were ver

## 4 — Convert and Save as a Hugging Face Model

The next step converts the workshop model into a Hugging Face-compatible artifact.

The save directory contains:

```text
config.json
modeling_workshop_gpt.py
model weights
```

`save_pretrained()` writes the model weights in Hugging Face's serialization format,
which uses SafeTensors when available.

The conversion also maps the original model state into the standalone model class
required by `trust_remote_code=True`.

In [15]:
import os, json, sys
from collections import OrderedDict
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/workshop/workshop-v1-hf"

# SAVE_DIR = "workshop-v1-hf"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── This standalone file gets saved next to the weights ───────────
#    (pure PyTorch, no torchtune — required for trust_remote_code)
MODELING_CODE = """\
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PretrainedConfig, PreTrainedModel

class WorkshopGPTConfig(PretrainedConfig):
    model_type = "workshop_gpt"
    def __init__(self, n_layer=12, n_head=12, n_embd=768, vocab_size=50304,
                 block_size=1024, n_inner=3072, rope_theta=10000.0, **kwargs):
        super().__init__(**kwargs)
        self.n_layer, self.n_head, self.n_embd = n_layer, n_head, n_embd
        self.vocab_size, self.block_size = vocab_size, block_size
        self.n_inner, self.rope_theta = n_inner, rope_theta

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).type_as(x) * self.scale

class RotaryPositionalEmbeddings(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000.0):
        super().__init__()
        self.dim, self.max_seq_len, self.base = dim, max_seq_len, base
        theta = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("theta", theta, persistent=False)
        self._build_cache(max_seq_len)
    def _build_cache(self, seq_len):
        seq = torch.arange(seq_len, device=self.theta.device)
        freqs = torch.outer(seq, self.theta)
        self.register_buffer("cache", torch.stack([freqs.cos(), freqs.sin()], dim=-1), persistent=False)
    def forward(self, x, *, input_pos=None):
        seq_len = x.shape[-2]
        if seq_len > self.cache.shape[0]: self._build_cache(seq_len)
        cache = self.cache[:seq_len] if input_pos is None else self.cache[input_pos]
        x1, x2 = x.float().unflatten(-1, (-1, 2)).unbind(-1)
        cos, sin = cache.unbind(-1)
        shape = [1] * (x.ndim - 2) + list(cos.shape)
        cos, sin = cos.view(*shape), sin.view(*shape)
        return torch.stack([x1*cos - x2*sin, x1*sin + x2*cos], dim=-1).flatten(-2).type_as(x)

class ReluSquaredMLP(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc_in  = nn.Linear(dim, hidden_dim, bias=False)
        self.fc_out = nn.Linear(hidden_dim, dim, bias=False)
    def forward(self, x):
        return self.fc_out(F.relu(self.fc_in(x)).square())

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, head_dim, rope):
        super().__init__()
        self.n_head, self.head_dim = n_head, head_dim
        self.q_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.k_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.v_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.output_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.rope = rope
    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = self.rope(q), self.rope(k)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.output_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        hd = config.n_embd // config.n_head
        rope = RotaryPositionalEmbeddings(hd, config.block_size, config.rope_theta)
        self.sa_norm  = RMSNorm(config.n_embd)
        self.attn     = CausalSelfAttention(config.n_embd, config.n_head, hd, rope)
        self.mlp_norm = RMSNorm(config.n_embd)
        self.mlp      = ReluSquaredMLP(config.n_embd, config.n_inner)
    def forward(self, x):
        x = x + self.attn(self.sa_norm(x))
        return x + self.mlp(self.mlp_norm(x))

class WorkshopGPTForCausalLM(PreTrainedModel):
    config_class = WorkshopGPTConfig
    def __init__(self, config):
        super().__init__(config)
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.n_embd)
        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.norm = RMSNorm(config.n_embd)
        self.output = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.output.weight = nn.Parameter(self.tok_embeddings.weight.clone())
    def forward(self, input_ids, **kwargs):
        x = self.tok_embeddings(input_ids)
        for layer in self.layers:
            x = layer(x)
        return type("Out", (), {"logits": self.output(self.norm(x))})()
"""

with open(os.path.join(SAVE_DIR, "modeling_workshop_gpt.py"), "w") as f:
    f.write(MODELING_CODE)

# ── config.json ───────────────────────────────────────────────────
json.dump({
    "model_type": "workshop_gpt",
    "architectures": ["WorkshopGPTForCausalLM"],
    "auto_map": {
        "AutoConfig": "modeling_workshop_gpt.WorkshopGPTConfig",
        "AutoModelForCausalLM": "modeling_workshop_gpt.WorkshopGPTForCausalLM",
    },
    "n_layer": 12, "n_head": 12, "n_embd": 768, "vocab_size": 50304,
    "block_size": 1024, "n_inner": 3072, "rope_theta": 10000.0,
    "torch_dtype": "bfloat16",
}, open(os.path.join(SAVE_DIR, "config.json"), "w"), indent=2)

# ── Map torchtune weights → HF and save ──────────────────────────
src = model._orig_mod if hasattr(model, "_orig_mod") else model
new_sd = OrderedDict((k, v.clone()) for k, v in src.state_dict().items())

sys.path.insert(0, SAVE_DIR)
from modeling_workshop_gpt import WorkshopGPTConfig, WorkshopGPTForCausalLM

hf_model = WorkshopGPTForCausalLM(WorkshopGPTConfig())
info = hf_model.load_state_dict(new_sd, strict=False)
print(f"Missing (expected — tied): {info.missing_keys}")
print(f"Unexpected: {info.unexpected_keys}")

hf_model.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}/:", os.listdir(SAVE_DIR))


Mounted at /content/drive
Missing (expected — tied): ['output.weight']
Unexpected: []


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/workshop/workshop-v1-hf/: ['modeling_workshop_gpt.py', 'config.json', '__pycache__', 'model.safetensors']


## 5 — Reload through Hugging Face

The converted model can then be loaded using:

```python
AutoModelForCausalLM.from_pretrained(...)
```

with `trust_remote_code=True` because the repository includes the custom workshop
model implementation.

This demonstrates a different loading interface while preserving the same learned
model behavior.

In [17]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "JustinAngel/workshop-v1-pretraining",
    trust_remote_code=True).cuda().eval()
enc = tiktoken.get_encoding("gpt2")

def generate(prompt):
  input_ids = torch.tensor([enc.encode(prompt)], device="cuda")
  output = model.generate(input_ids, max_new_tokens=100, do_sample=True, temperature=0.8, top_k=50)
  print(enc.decode(output[0].tolist()))


for prompt in ["The meaning of life is", "Once upon a time there was"]:
    print(f">> {prompt}")
    print(generate(prompt))
    print()

Loading weights:   0%|          | 0/98 [00:00<?, ?it/s]

[transformers] WorkshopGPTForCausalLM LOAD REPORT from: JustinAngel/workshop-v1-pretraining
Key           | Status     |  | 
--------------+------------+--+-
output.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


>> The meaning of life is
The meaning of life is that we are all living in a state of equilibrium.
The human mind is the highest point of equilibrium in the universe, but it is dependent on the individual’s ability to adapt. If we are to live in a state of equilibrium, we must be able to adapt to the ever-changing environment. However, when we are in equilibrium, we are constantly exposed to a series of “noise-based” stimuli. These stimuli cause us to adapt, and the person perce
None

>> Once upon a time there was
Once upon a time there was a serious illness called a “epilepsy”. At the time this was a serious medical condition, and doctors were not sure if it was a disease that was causing the death of the child.
In the late 1950s, the American Academy of Pediatrics (AAP) released a new version of the “Epilepsy Report” which is still used today. The report was issued by the American Academy of Pediatrics (AAP) in the early 1960s. It is
None



---

## Observations

- Saving learned parameters separates model state from the lifetime of a notebook runtime.
- A `.pth` checkpoint can restore the trained state after the architecture is reconstructed.
- Successful loading should be followed by an inference check rather than assuming that file loading alone proves correctness.
- The recorded `.pth` load restored **123,587,328 parameters**.
- The workshop model can be repackaged into a Hugging Face-compatible format and saved with SafeTensors.
- Hugging Face loading provides a reusable deployment/sharing interface around the same learned weights.
- Checkpoint formats and deployment formats solve different parts of the model lifecycle.

## Connection to the Previous Unit

```text
Unit 08
Training updates the parameters
        ↓
Unit 09
Saving preserves those parameters
        ↓
Loading restores them for inference or continued work
```

The combined lifecycle is:

```text
Build → Train → Save → Load → Reuse
```